In [ ]:
# The recipe is configured for two Kaggle T4 GPUs.
!nvidia-smi
import torch

gpu_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
print(f"GPU count: {len(gpu_names)}")
print("GPUs:", gpu_names)
if len(gpu_names) != 2 or any("T4" not in name for name in gpu_names):
    raise RuntimeError(f"Select Kaggle GPU T4 x2; allocated hardware is {gpu_names}.")


In [ ]:
# Use the exact repository commit that generated this notebook.
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
import subprocess

os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!git clone --filter=blob:none --no-checkout https://github.com/spirlness/Automodel.git Automodel
os.chdir("/kaggle/working/Automodel")
!git fetch --depth 1 origin 84552b22a
!git checkout --detach 84552b22a
checked_out_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if not checked_out_commit.startswith("84552b22a"):
    raise RuntimeError(f"Expected source commit 84552b22a, got {checked_out_commit}")
print(f"Using source commit {checked_out_commit}")
!uv sync --locked --no-default-groups --inexact

# FineWeb is public. If the Kaggle environment provides HF_TOKEN, the
# processor will use it; an HF Secret is not required for this run.
if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is available")
else:
    print("HF_TOKEN is not set; using the public FineWeb dataset")


In [ ]:
# Verify Hub access, tokenization, and exact binary writing first.
!uv run python projects/gpt2_fineweb_500m/tools/nanogpt_data_processor.py \
  --dataset HuggingFaceFW/fineweb \
  --set-name sample-10BT \
  --output-dir /kaggle/working/fineweb_smoke \
  --max-tokens 1M \
  --max-length 1024


In [ ]:
# Construct the exact model and loss from the Kaggle YAML before torchrun.
from nemo_automodel.components.config._arg_parser import parse_args_and_load_config

cfg = parse_args_and_load_config("/kaggle/working/Automodel/projects/gpt2_fineweb_500m/config/gpt2_fineweb_t4x2.yaml")
model = cfg.model.instantiate()
loss = cfg.loss_fn.instantiate()
assert model.__class__.__name__ == "GPT2LMHeadModel"
assert loss.__class__.__name__ == "MaskedCrossEntropy"
assert cfg.optimizer._target_.__module__ == "torch.optim"
assert cfg.optimizer._target_.__name__ == "AdamW"
assert cfg.distributed.strategy == "ddp"
print(f"Config OK: model={model.__class__.__name__}, loss={loss.__class__.__name__}, strategy={cfg.distributed.strategy}")
del model, loss, cfg


In [ ]:
# Run one baseline DDP + AdamW training step on the smoke dataset.
!uv run automodel /kaggle/working/Automodel/projects/gpt2_fineweb_500m/config/gpt2_fineweb_t4x2.yaml \
  --nproc-per-node 2 \
  --dataset.file_pattern=/kaggle/working/fineweb_smoke_max_tokens_1M/dataset.bin \
  --step_scheduler.global_batch_size=8 \
  --step_scheduler.local_batch_size=4 \
  --step_scheduler.max_steps=1 \
  --step_scheduler.ckpt_every_steps=1000000 \
  --step_scheduler.val_every_steps=1000000 \
  --step_scheduler.save_checkpoint_every_epoch=false \
  --checkpoint.enabled=false


In [ ]:
# The smoke test passed, so build the 1B-token dataset and train.
!uv run python projects/gpt2_fineweb_500m/tools/nanogpt_data_processor.py \
  --dataset HuggingFaceFW/fineweb \
  --set-name sample-10BT \
  --output-dir /kaggle/working/fineweb_1B \
  --max-tokens 1B \
  --max-length 1024
!uv run automodel /kaggle/working/Automodel/projects/gpt2_fineweb_500m/config/gpt2_fineweb_t4x2.yaml \
  --nproc-per-node 2 \
  --dataset.file_pattern=/kaggle/working/fineweb_1B_max_tokens_1B/dataset.bin \
  --step_scheduler.global_batch_size=8 \
  --step_scheduler.local_batch_size=4 \
  --step_scheduler.max_steps=122070 \
  --step_scheduler.ckpt_every_steps=10000 \
  --step_scheduler.val_every_steps=1000000 \
  --step_scheduler.save_checkpoint_every_epoch=false \
  --checkpoint.checkpoint_dir=/kaggle/working/checkpoints \
  --checkpoint.max_recent_checkpoints=3
